# V2a-RSN 220127_F4_run2 — Probabilistic State Notebook: Switching VAR

Prerequisite: the `switching_var` connectivity stage must already exist for the selected `RUN_ID`.
Optional descriptive HMM fit, starting from one state. Duration diagnostics can motivate later models; the fit does not establish neural regimes and is separate from prospective state selection.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


In [ ]:
from effectome.dynamics import ProbabilisticStateConfig
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_probabilistic_states,
)

DATASET_ID = "220127_F4_run2"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220127_F4_run2"
RUN_ID = "reference"
METHOD = "switching_var"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
FORCE = False

PROBABILISTIC_CFG = ProbabilisticStateConfig(
    n_states=1,
    n_components=8,
    standardize=True,
    covariance_floor=1.0e-4,
    n_iter=50,
    tol=1.0e-4,
    seed=42,
)

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} probabilistic states",
    expected_stages=["probabilistic_states"],
    force=FORCE,
)


In [ ]:
probabilistic = run_probabilistic_states(run, probabilistic_cfg=PROBABILISTIC_CFG, force=FORCE)
print(probabilistic.labels.shape)
print(probabilistic.log_likelihood)
print(probabilistic.hsmm_candidate.status)
print(probabilistic.hsmm_candidate.non_geometric_states)
print_stage_status(run)
